#  Preprocessing Pipeline Notebook

This notebook builds a **general, reusable preprocessing pipeline** for datasets.
It ensures raw data is consistently transformed into clean train/test sets ready for machine learning.

### Workflow Overview
1. **Load dataset** -> `load()` function supports CSV, JSON, and Excel formats.
2. **Preprocess data** -> `preprocess_data()` scales numeric columns, encodes categorical columns, separates target, and splits into train/test sets.
3. **Validation checks** -> `validation_report()` confirms shapes, target distribution, and scaling ranges.
4. **Config-driven** -> pipeline behavior is controlled by a flexible `config` dictionary (scaling method, columns, target, split size).
5. **Outputs** -> clean `X_train, X_test, y_train, y_test` sets, optionally saved for modeling.

This notebook provides a **stable, modular pipeline** that can handle multiple file formats and produce validated datasets for end-to-end testing.


In [1]:
import pandas as pd
import json

def load(file_path):
    """
    Loads a dataset from CSV, JSON, or Excel into a pandas DataFrame.

    Parameters:
        file_path (str): Path to the dataset file

    Returns:
        DataFrame: Loaded dataset
    """
    if file_path.endswith(".csv"):
        df = pd.read_csv(file_path)
    elif file_path.endswith(".json"):
        with open(file_path, "r") as f:
            data = json.load(f)
        df = pd.DataFrame(data)
    elif file_path.endswith(".xlsx") or file_path.endswith(".xls"):
        df = pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported file format. Please use CSV, JSON, or Excel.")

    return df



The `load()` function reads a dataset file and converts it into a **pandas DataFrame** for further preprocessing.

- **CSV files** -> loaded using `pd.read_csv()`
- **JSON files** -> opened with `json.load()` and converted to DataFrame
- **Excel files (.xlsx / .xls)** -> loaded using `pd.read_excel()`
- **Error handling** -> raises a clear message if the format is unsupported

 `load()` is the **entry point** of the pipeline.
It ensures that no matter the dataset format (CSV, JSON, Excel), you always get a clean DataFrame ready for preprocessing.


In [2]:
# Define Preprocessing Pipeline Function

from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

def preprocess_data(df, config):
    """
    Preprocesses the dataset based on configuration.

    Parameters:
        df (DataFrame): Input dataset
        config (dict): Configuration dictionary with keys:
            - scale_cols: list of numeric columns to scale
            - encode_cols: list of categorical columns to encode
            - target_col: target column name
            - test_size: fraction for test split (default 0.2)
            - stratify: whether to stratify split (default True)
            - scaling: 'zscore' or 'minmax'

    Returns:
        X_train, X_test, y_train, y_test
    """

    # Copy dataset
    data = df.copy()

    # Scaling
    if config.get("scaling") == "zscore":
        scaler = StandardScaler()
    else:
        scaler = MinMaxScaler()

    data[config["scale_cols"]] = scaler.fit_transform(data[config["scale_cols"]])

    # Encoding categorical columns
    for col in config["encode_cols"]:
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col])

    # Split features and target
    X = data.drop(columns=[config["target_col"]])
    y = data[config["target_col"]]

    # Train/test split
    stratify = y if config.get("stratify", True) else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config.get("test_size", 0.2), stratify=stratify, random_state=42
    )

    return X_train, X_test, y_train, y_test


The `preprocess_data()` function prepares the dataset for modeling based on the configuration provided.

- **Scaling** -> applies either Z‑score (`StandardScaler`) or Min‑Max (`MinMaxScaler`) to numeric columns listed in `scale_cols`.
- **Encoding** -> converts categorical columns listed in `encode_cols` into numeric values using `LabelEncoder`.
- **Target separation** -> splits the dataset into features (`X`) and target (`y`) based on `target_col`.
- **Train/Test split** -> divides the data into training and testing sets, preserving class distribution if `stratify=True`.

`preprocess_data()` transforms raw data into **clean, scaled, encoded, and split datasets** (`X_train, X_test, y_train, y_test`) ready for machine learning models.


In [6]:
# Step 2: Define Configuration for Dataset

config = {
    "scale_cols": ["frp"],          # numeric column to scale
    "encode_cols": ["satellite"],   # categorical column to encode
    "target_col": "confidence",     # target variable
    "test_size": 0.2,               # 80/20 split
    "stratify": True,               # preserve class distribution
    "scaling": "minmax"             # choose 'minmax' or 'zscore'
}



The `config` dictionary defines how the preprocessing pipeline should handle the dataset.

- **scale_cols** -> numeric columns to scale (here, `frp`)
- **encode_cols** -> categorical columns to encode (here, `satellite`)
- **target_col** -> the column to predict (here, `confidence`)
- **test_size** -> fraction of data reserved for testing (0.2 = 20%)
- **stratify** -> ensures train/test split preserves target distribution
- **scaling** -> method for scaling numeric values (`minmax` or `zscore`)

`config` makes the pipeline **flexible and reusable**.
By changing values in this dictionary, the same notebook can preprocess different datasets without rewriting code.


In [8]:
# Apply Preprocessing Pipeline

# Load dataset
df = load("modis_2024_India.csv")   # or "modis_fire.json"

X_train, X_test, y_train, y_test = preprocess_data(df, config)

# Quick validation checks
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Target distribution in train:\n", y_train.value_counts(normalize=True))
print("Target distribution in test:\n", y_test.value_counts(normalize=True))
print("Scaled FRP range (train):", X_train["frp"].min(), "to", X_train["frp"].max())


Train shape: (59223, 14)
Test shape: (14806, 14)
Target distribution in train:
 confidence
67    0.027608
66    0.027388
65    0.026729
63    0.026510
64    0.026493
        ...   
4     0.000152
6     0.000101
2     0.000051
3     0.000051
1     0.000051
Name: proportion, Length: 101, dtype: float64
Target distribution in test:
 confidence
67    0.027624
66    0.027354
65    0.026746
63    0.026543
64    0.026476
        ...   
7     0.000203
4     0.000135
6     0.000135
5     0.000135
2     0.000068
Name: proportion, Length: 99, dtype: float64
Scaled FRP range (train): 0.0 to 0.673945673108853


###  Applying the Preprocessing Pipeline
This step runs the complete workflow:

1. **Load dataset** -> `load("modis_2024_India.csv")` reads the raw file into a DataFrame.
2. **Preprocess data** -> `preprocess_data(df, config)` applies scaling, encoding, and train/test splitting based on the config dictionary.
3. **Validation checks** -> prints:
   - Train/Test shapes (to confirm split size).
   - Target distribution in train/test (to confirm stratification).
   - Scaled FRP range (to confirm numeric scaling).

This block demonstrates the pipeline in action, showing that the dataset is correctly transformed and ready for modeling.


In [10]:
def validation_report(X_train, X_test, y_train, y_test, config):
    print("Train shape:", X_train.shape)
    print("Test shape:", X_test.shape)
    print("\nTarget distribution in train:\n", y_train.value_counts(normalize=True).head(10))
    print("\nTarget distribution in test:\n", y_test.value_counts(normalize=True).head(10))

    for col in config["scale_cols"]:
        print(f"\nScaled {col} range (train): {X_train[col].min()} to {X_train[col].max()}")


validation_report(X_train,X_test, y_train, y_test, config)

Train shape: (59223, 14)
Test shape: (14806, 14)

Target distribution in train:
 confidence
67    0.027608
66    0.027388
65    0.026729
63    0.026510
64    0.026493
68    0.025683
69    0.025649
62    0.025581
61    0.024855
70    0.024754
Name: proportion, dtype: float64

Target distribution in test:
 confidence
67    0.027624
66    0.027354
65    0.026746
63    0.026543
64    0.026476
69    0.025665
68    0.025665
62    0.025598
61    0.024855
70    0.024787
Name: proportion, dtype: float64

Scaled frp range (train): 0.0 to 0.673945673108853


The `validation_report()` function provides a quick summary to check if preprocessing worked correctly.

- **Train/Test shapes** -> confirms dataset split size.
- **Target distribution** -> shows top 10 confidence values in train/test to verify stratification.
- **Scaled ranges** -> prints min/max values of numeric columns listed in `scale_cols` to confirm scaling.

`validation_report()` acts as a **sanity check**.
It ensures the processed data is balanced, correctly scaled, and ready for modeling.
